In [1]:
import pandas as pd
import pickle

# Load train_dataset 
train_data = pd.read_csv('datasets/train_dataset.csv')

# Separate features and target
X = train_data.drop(['isFraud'], axis=1)
y = train_data['isFraud']

# Load the 10-fold indices
with open("10_fold_indices.pkl", "rb") as f:
    splits = pickle.load(f)

print(f"Loaded {len(splits)} folds.")

Loaded 10 folds.


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import StratifiedKFold
from category_encoders import TargetEncoder
import os
import pickle

# -------------------- LightGBM Parameters --------------------
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'random_state': 42
}

# -------------------- Setup --------------------
auc_scores = []
accuracy_scores = []
recall_0, recall_1 = [], []
precision_0, precision_1 = [], []
f1_0, f1_1 = [], []
feature_importance_df = pd.DataFrame()

os.makedirs('models/lightgbm', exist_ok=True)
os.makedirs('feature_importance', exist_ok=True)

# Assume X and y are already loaded
categorical_cols = X.select_dtypes(include='object').columns.tolist()

# -------------------- Cross-Validation Loop --------------------
for fold, (train_idx, val_idx) in enumerate(splits, 1):
    print(f"\n Fold {fold}")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Target Encoding
    encoder = TargetEncoder(cols=categorical_cols)
    X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols], y_train)
    X_val[categorical_cols] = encoder.transform(X_val[categorical_cols])

    # LightGBM Dataset
    train_dataset = lgb.Dataset(X_train, label=y_train)
    val_dataset = lgb.Dataset(X_val, label=y_val)

    # Train
    model = lgb.train(
        params=params,
        train_set=train_dataset,
        valid_sets=[val_dataset],
        num_boost_round=5000,
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(50)
        ]
    )

    # Predict and Evaluate
    y_val_pred_prob = model.predict(X_val, num_iteration=model.best_iteration)
    y_val_pred = (y_val_pred_prob > 0.5).astype(int)

    auc = roc_auc_score(y_val, y_val_pred_prob)
    accuracy = accuracy_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred, average=None, zero_division=0)
    recall = recall_score(y_val, y_val_pred, average=None)
    f1 = f1_score(y_val, y_val_pred, average=None)

    # Append metrics
    auc_scores.append(auc)
    accuracy_scores.append(accuracy)
    precision_0.append(precision[0])
    precision_1.append(precision[1])
    recall_0.append(recall[0])
    recall_1.append(recall[1])
    f1_0.append(f1[0])
    f1_1.append(f1[1])

    print(f" Fold {fold} AUC-ROC : {auc:.4f} | Best Iteration: {model.best_iteration}")
    print(f"   Accuracy    : {accuracy:.4f}")
    print(f"   Precision_0 : {precision[0]:.4f} | Precision_1 : {precision[1]:.4f}")
    print(f"   Recall_0    : {recall[0]:.4f} | Recall_1    : {recall[1]:.4f}")
    print(f"   F1 Score_0  : {f1[0]:.4f} | F1 Score_1  : {f1[1]:.4f}")

    # Save model
    model.save_model(f'models/lightgbm/fold_{fold}.txt')

    # Feature importance
    fold_importance = pd.DataFrame({
        'feature': model.feature_name(),
        f'fold_{fold}': model.feature_importance(importance_type='gain')
    })
    if feature_importance_df.empty:
        feature_importance_df = fold_importance
    else:
        feature_importance_df = feature_importance_df.merge(fold_importance, on='feature')

# -------------------- Final Summary --------------------
feature_importance_df['avg_importance'] = feature_importance_df.drop('feature', axis=1).mean(axis=1)
feature_importance_df.sort_values(by='avg_importance', ascending=False, inplace=True)
feature_importance_df.to_csv('feature_importance/lgbm_feature_importance_target_encoded.csv', index=False)

# Final metrics summary
print("\nCross-Validation Results (LightGBM)")
for i in range(len(auc_scores)):
    print(f"Fold {i+1}: AUC={auc_scores[i]:.4f} | Acc={accuracy_scores[i]:.4f} | "
          f"Prec_0={precision_0[i]:.4f} | Prec_1={precision_1[i]:.4f} | "
          f"Recall_0={recall_0[i]:.4f} | Recall_1={recall_1[i]:.4f} | "
          f"F1_0={f1_0[i]:.4f} | F1_1={f1_1[i]:.4f}")

print("\nFinal Averages:")
print(f"AUC-ROC      : {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Accuracy     : {np.mean(accuracy_scores):.4f}")
print(f"Precision_0  : {np.mean(precision_0):.4f}")
print(f"Precision_1  : {np.mean(precision_1):.4f}")
print(f"Recall_0     : {np.mean(recall_0):.4f}")
print(f"Recall_1     : {np.mean(recall_1):.4f}")
print(f"F1 Score_0   : {np.mean(f1_0):.4f}")
print(f"F1 Score_1   : {np.mean(f1_1):.4f}")



 Fold 1
Training until validation scores don't improve for 50 rounds
[50]	valid_0's auc: 0.89871
[100]	valid_0's auc: 0.917034
[150]	valid_0's auc: 0.926482
[200]	valid_0's auc: 0.931943
[250]	valid_0's auc: 0.935693
[300]	valid_0's auc: 0.939356
[350]	valid_0's auc: 0.941742
[400]	valid_0's auc: 0.943676
[450]	valid_0's auc: 0.94557
[500]	valid_0's auc: 0.947164
[550]	valid_0's auc: 0.948031
[600]	valid_0's auc: 0.949135
[650]	valid_0's auc: 0.950437
[700]	valid_0's auc: 0.951319
[750]	valid_0's auc: 0.953254
[800]	valid_0's auc: 0.954056
[850]	valid_0's auc: 0.954745
[900]	valid_0's auc: 0.955595
[950]	valid_0's auc: 0.955958
[1000]	valid_0's auc: 0.956342
[1050]	valid_0's auc: 0.957228
[1100]	valid_0's auc: 0.958175
[1150]	valid_0's auc: 0.95858
[1200]	valid_0's auc: 0.95911
[1250]	valid_0's auc: 0.959645
[1300]	valid_0's auc: 0.960064
[1350]	valid_0's auc: 0.960573
[1400]	valid_0's auc: 0.961007
[1450]	valid_0's auc: 0.961381
[1500]	valid_0's auc: 0.961895
[1550]	valid_0's auc: 0.